这里给了使用 ms-swift 多机多卡实现 QianFanOCR GRPO 训练，QianFanOCR 视觉模块采用 Internvl3.5，语言模块采用 Qwen3

```shell
.
├── rollout.sh                          # reference model rollout
├── run_qianfanocr_grpo_table_node1.sh  # node1 training script
└── run_qianfanocr_grpo_table_node2.sh  # node2 training script
```

```
--model 指定 QianfanOCR huggingface 权重路径，推荐绝对路径
--external_plugins 指定奖励函数脚本，推荐绝对路径
--dataset 指定数据路径，推荐绝对路径
--output_dir 指定模型保存路径，推荐绝对路径
```

# Dataset
采用 `jsonl` 格式，每条数据格式如下，`solution` 代表 gt
```
{
    "messages":
    [
        {
            "role": "user",
            "content": "<image>your prompt like parse image to markdown"
        }
    ],
    "solution": "# 经山东省中小学教材审定委员会审查通过",
    "images":
    [
        "/mnt/dataset/images/page_001.jpg"
    ]
}
```

# Reward

`qianfanocr_table_reward_plugin_html.py`
当前 reward 主要优化模型对表格的解析能力，逻辑如下
- 取出 pred 和 gt 的所有表格，如果二者的表格数量不一致，难以进行表格匹配，直接计算pred和gt编辑距离相似度作为最终奖励；
- pred 和 gt 的表格数量相等，表格之间计算teds作为每个表格的奖励，然后去除表格，计算剩余文本编辑距离相似度得到剩余文本奖励，最终计算表格奖励和剩余文本奖励的均值作为最终奖励；

训练日志如下
```
Train:   0%|          | 0/13644 [00:00<?, ?it/s]
{'loss': 7.76727057, 'grad_norm': 199.721992, 'learning_rate': 1e-08, 'completions/mean_length': 1021.203125, 'completions/min_length': 52.0, 'completions/max_length': 2362.0, 'completions/clipped_ratio': 0.0, 'reward': 0.89361554, 'reward_std': 0.05483468, 'frac_reward_zero_std': 0.0, 'rewards/TableQianfanOCRDistance/mean': 0.89361548, 'rewards/TableQianfanOCRDistance/std': 0.08555585, 'kl': 1783.09606659, 'clip_ratio/low_mean': 0.01226649, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.02155004, 'clip_ratio/high_max': 0.1056338, 'clip_ratio/region_mean': 0.03381653, 'epoch': 0.0, 'global_step/max_steps': '1/13644', 'elapsed_time': '3m 8s', 'remaining_time': '29d 16h 28m 40s', 'memory(GiB)': 12.1, 'train_speed(s/it)': 188.002617}
{'loss': 4.22127295, 'grad_norm': 444.44157949, 'learning_rate': 3e-08, 'completions/mean_length': 1086.796875, 'completions/min_length': 528.0, 'completions/max_length': 2518.0, 'completions/clipped_ratio': 0.0, 'reward': 0.67562068, 'reward_std': 0.04701808, 'frac_reward_zero_std': 0.25, 'rewards/TableQianfanOCRDistance/mean': 0.67562068, 'rewards/TableQianfanOCRDistance/std': 0.41502681, 'kl': 2625.47730651, 'clip_ratio/low_mean': 0.00319046, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.00362223, 'clip_ratio/high_max': 0.02152438, 'clip_ratio/region_mean': 0.00681268, 'epoch': 0.0, 'global_step/max_steps': '2/13644', 'elapsed_time': '4m 24s', 'remaining_time': '20d 19h 59m 60s', 'memory(GiB)': 12.4, 'train_speed(s/it)': 131.945449}
{'loss': 3.42036581, 'grad_norm': 20.13026038, 'learning_rate': 4e-08, 'completions/mean_length': 3223.0, 'completions/min_length': 747.0, 'completions/max_length': 28818.0, 'completions/clipped_ratio': 0.0625, 'reward': 0.72528118, 'reward_std': 0.10050126, 'frac_reward_zero_std': 0.0, 'rewards/TableQianfanOCRDistance/mean': 0.72528118, 'rewards/TableQianfanOCRDistance/std': 0.42824182, 'kl': 77.26320148, 'clip_ratio/low_mean': 0.00906659, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.00596549, 'clip_ratio/high_max': 0.02453024, 'clip_ratio/region_mean': 0.01503208, 'epoch': 0.0, 'global_step/max_steps': '3/13644', 'elapsed_time': '13m 57s', 'remaining_time': '44d 1h 26m 53s', 'memory(GiB)': 18.39, 'train_speed(s/it)': 279.071409}
```